In [0]:
from pyspark.sql.functions import current_timestamp, col, monotonically_increasing_id

# Config
source_path = "/Volumes/assesment/bronze/raw_data/ewevo2022.csv"   # exact file, not the folder
bronze = "assesment.bronze.ons_very_old_population_raw"

# Task 2: read the raw CSV exactly as delivered
df_raw = (
    spark.read.format("csv")
    .option("inferSchema", True)   # detect column types from the values
    .option("sep", ",")            # comma-delimited
    .option("header", False)       # row 1 is a title, so columns become _c0.._c20
    .load(source_path)
)

display(df_raw)

# Bronze: save the raw rows unchanged, plus 3 audit columns
df_bronze = (
    df_raw
    .withColumn("_source_row_id", monotonically_increasing_id())   # original row order (tables don't keep order)
    .withColumn("_source_file", col("_metadata.file_path"))        # which file the data came from
    .withColumn("_ingested_at", current_timestamp())               # when it was loaded
)

df_bronze.write.format("delta").mode("overwrite").saveAsTable(bronze)
print(f"Saved {df_bronze.count()} rows to {bronze}")